In [13]:
import warnings
import os
import numpy as np
import pandas as pd
import xgboost as xgb
import catboost as cb
import joblib
from scipy.optimize import minimize
from sklearn.metrics import mean_squared_error

# --- Global Constants ---
PREDS_PATH = './model_predictions/'
DATA_PATH = './'
MODELS_SAVE_PATH = './final_models_v1/'
PREVIOUS_BEST_SCORE = 290912.17 # The score from the single XGBoost meta-model

# --- Winkler Score Helper Function ---
def winkler_score(y_true, lower, upper, alpha=0.1, return_coverage=False):
    width = upper - lower
    penalty_lower = np.where(y_true < lower, (lower - y_true) * (2 / alpha), 0)
    penalty_upper = np.where(y_true > upper, (y_true - upper) * (2 / alpha), 0)
    score = np.mean(width + penalty_lower + penalty_upper)
    
    if return_coverage:
        coverage = np.mean((y_true >= lower) & (y_true <= upper))
        return score, coverage, np.mean(width)
    return score

warnings.filterwarnings('ignore')
print("Libraries and helper functions loaded successfully.")

Libraries and helper functions loaded successfully.


In [14]:
print("--- Loading all pre-trained base model predictions and raw data ---")

try:
    # Load Mean Model Predictions (OOF and Test)
    oof_xgb_preds = np.load(f'{PREDS_PATH}oof_xgb_preds.npy')
    test_xgb_preds = np.load(f'{PREDS_PATH}test_xgb_preds.npy')
    oof_cb_preds = np.load(f'{PREDS_PATH}oof_cb_preds.npy')
    test_cb_preds = np.load(f'{PREDS_PATH}test_cb_preds.npy')
    oof_nn_preds = np.load(f'{PREDS_PATH}oof_nn_preds.npy')
    test_nn_preds = np.load(f'{PREDS_PATH}test_nn_preds.npy')
    print("Base model predictions loaded.")
    
    # Load Ground Truth and Raw Feature Data
    y_true = pd.read_csv(DATA_PATH + 'dataset.csv')['sale_price']
    df_train_raw = pd.read_csv(DATA_PATH + 'dataset.csv')
    df_test_raw = pd.read_csv(DATA_PATH + 'test.csv')
    print("Ground truth and raw feature data loaded.")

except FileNotFoundError as e:
    print(f"\nERROR: Could not find a required file. {e}")
    print("Please ensure all prediction and data files are in the correct directories.")

--- Loading all pre-trained base model predictions and raw data ---
Base model predictions loaded.
Ground truth and raw feature data loaded.


In [15]:
print("\n--- Re-creating the Enhanced Meta-Feature Set ---")

# --- 1. Recreate the Super-Ensemble Mean Predictions ---
oof_preds_stack_mean = np.vstack([oof_xgb_preds, oof_cb_preds, oof_nn_preds]).T
test_preds_stack_mean = np.vstack([test_xgb_preds, test_cb_preds, test_nn_preds]).T

def get_ensemble_rmse(weights):
    final_prediction = np.dot(oof_preds_stack_mean, weights)
    return np.sqrt(mean_squared_error(y_true, final_prediction))

result_mean = minimize(get_ensemble_rmse, [1/3]*3, method='SLSQP', bounds=[(0,1)]*3, constraints=({'type': 'eq', 'fun': lambda w: 1 - sum(w)}))
best_mean_weights = result_mean.x
oof_ensemble_mean = np.dot(oof_preds_stack_mean, best_mean_weights)
test_ensemble_mean = np.dot(test_preds_stack_mean, best_mean_weights)
print("Super-ensemble mean predictions recreated.")

# --- 2. Create Context Features for Train and Test Sets ---
def create_context_features(df):
    df = df.rename(columns={'latitude': 'lat', 'longitude': 'long', 'year_built': 'yr_built', 'year_reno':'yr_renovated'})
    context_cols = ['grade', 'sqft', 'lat', 'long', 'yr_built', 'yr_renovated']
    X_context = df[context_cols].fillna(0)
    X_context['property_age'] = 2025 - X_context['yr_built']
    return X_context

X_context_train = create_context_features(df_train_raw)
X_context_test = create_context_features(df_test_raw)
print("Context features created for train and test sets.")

# --- 3. Combine into Final Enhanced Meta-Feature Sets ---
X_meta_train_enhanced = pd.concat([pd.DataFrame({'xgb_pred': oof_xgb_preds, 'cb_pred': oof_cb_preds, 'nn_pred': oof_nn_preds, 'ensemble_mean': oof_ensemble_mean}), X_context_train.reset_index(drop=True)], axis=1)
X_meta_test_enhanced = pd.concat([pd.DataFrame({'xgb_pred': test_xgb_preds, 'cb_pred': test_cb_preds, 'nn_pred': test_nn_preds, 'ensemble_mean': test_ensemble_mean}), X_context_test.reset_index(drop=True)], axis=1)

print(f"Final meta-feature sets created. Train shape: {X_meta_train_enhanced.shape}, Test shape: {X_meta_test_enhanced.shape}")


--- Re-creating the Enhanced Meta-Feature Set ---
Super-ensemble mean predictions recreated.
Context features created for train and test sets.
Final meta-feature sets created. Train shape: (200000, 11), Test shape: (200000, 11)


In [16]:
print("\n--- Loading all 4 pre-trained meta-models ---")

try:
    # Load XGBoost Meta-Models
    model_xgb_lower = joblib.load(os.path.join(MODELS_SAVE_PATH, 'meta_model_xg_final_lower.joblib'))
    model_xgb_upper = joblib.load(os.path.join(MODELS_SAVE_PATH, 'meta_model_xg_final_upper.joblib'))
    print("XGBoost meta-models loaded.")

    # Load CatBoost Meta-Models
    model_cb_lower = joblib.load(os.path.join(MODELS_SAVE_PATH, 'meta_model_catboost_lower.joblib'))
    model_cb_upper = joblib.load(os.path.join(MODELS_SAVE_PATH, 'meta_model_catboost_upper.joblib'))
    print("CatBoost meta-models loaded.")
except FileNotFoundError as e:
    print(f"\nERROR: Could not find a saved meta-model file. {e}")
    print("Please ensure the meta-models were trained and saved correctly.")

# --- Generate OOF predictions from all meta-models ---
print("\n--- Generating OOF predictions for blending optimization ---")
oof_xgb_lower = model_xgb_lower.predict(X_meta_train_enhanced)
oof_xgb_upper = model_xgb_upper.predict(X_meta_train_enhanced)
oof_cb_lower = model_cb_lower.predict(X_meta_train_enhanced)
oof_cb_upper = model_cb_upper.predict(X_meta_train_enhanced)


--- Loading all 4 pre-trained meta-models ---
XGBoost meta-models loaded.
CatBoost meta-models loaded.

--- Generating OOF predictions for blending optimization ---


In [17]:
print("\n--- Searching for the Optimal Blend of the Two Champion Meta-Models ---")

def get_winkler_from_blend(weights):
    w_xgb = weights[0]
    w_cb = 1 - w_xgb # Ensures weights sum to 1
    
    # Create the blended interval predictions
    final_lower = w_xgb * oof_xgb_lower + w_cb * oof_cb_lower
    final_upper = w_xgb * oof_xgb_upper + w_cb * oof_cb_upper
    
    return winkler_score(y_true, final_lower, final_upper)

# Run the optimizer to find the best weight for the XGBoost model
result_blend = minimize(get_winkler_from_blend, [0.5], method='L-BFGS-B', bounds=[(0,1)])
best_xgb_weight = result_blend.x[0]
best_cb_weight = 1 - best_xgb_weight
best_final_score = result_blend.fun

# --- THE ULTIMATE PIPELINE: FINAL SHOWDOWN ---
print("\n" + "="*60)
print("     THE TWO-CHAMPION META-MODEL PIPELINE: FINAL SHOWDOWN")
print("="*60)
print(f"Previous Best (Single Meta-Model) : ${PREVIOUS_BEST_SCORE:,.2f}")
print(f"New ENSEMBLED META-MODEL Score    : ${best_final_score:,.2f}")
print(f" (Optimal Weights: XGB={best_xgb_weight:.4f}, CB={best_cb_weight:.4f})")
print("="*60)


--- Searching for the Optimal Blend of the Two Champion Meta-Models ---

     THE TWO-CHAMPION META-MODEL PIPELINE: FINAL SHOWDOWN
Previous Best (Single Meta-Model) : $290,912.17
New ENSEMBLED META-MODEL Score    : $284,269.57
 (Optimal Weights: XGB=0.0000, CB=1.0000)


In [18]:
# --- Create Final Submission File ---
print("\n--- Creating final submission file with optimally blended predictions ---")

# 1. Generate test predictions from all meta-models
test_xgb_lower = model_xgb_lower.predict(X_meta_test_enhanced)
test_xgb_upper = model_xgb_upper.predict(X_meta_test_enhanced)
test_cb_lower = model_cb_lower.predict(X_meta_test_enhanced)
test_cb_upper = model_cb_upper.predict(X_meta_test_enhanced)

# 2. Blend the test predictions using the optimal weights
final_test_lower = best_xgb_weight * test_xgb_lower + best_cb_weight * test_cb_lower
final_test_upper = best_xgb_weight * test_xgb_upper + best_cb_weight * test_cb_upper

# 3. Create and save the submission DataFrame
submission_df = pd.DataFrame({'id': df_test_raw['id'], 'pi_lower': final_test_lower, 'pi_upper': final_test_upper})
submission_df['pi_lower'] = submission_df['pi_lower'].clip(0, None)
submission_df['pi_upper'] = np.maximum(submission_df['pi_lower'], submission_df['pi_upper'])

submission_filename = f'submission_final_blend_{int(best_final_score)}.csv'
submission_df.to_csv(submission_filename, index=False)

print(f"\n'{submission_filename}' created successfully! Good luck on the leaderboard!")
display(submission_df.head())


--- Creating final submission file with optimally blended predictions ---

'submission_final_blend_284269.csv' created successfully! Good luck on the leaderboard!


,id,pi_lower,pi_upper
0,200000,828813.263100,1.077837e+06
1,200001,573899.514685,7.812186e+05
2,200002,458082.063233,6.391792e+05
3,200003,292426.483054,4.179944e+05
4,200004,368525.046278,7.360905e+05
